<a href="https://colab.research.google.com/github/prroud/AI_learning_journey/blob/main/Cassava_Leaf_Disease_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
!pip install torchmetrics

In [69]:
import torch
from torch import nn
from torchvision import models, transforms, datasets
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import os
import copy
from torchmetrics.classification import Accuracy
from sklearn.model_selection import train_test_split

In [70]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nirmalsankalana/cassava-leaf-disease-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cassava-leaf-disease-classification' dataset.
Path to dataset files: /kaggle/input/cassava-leaf-disease-classification


In [71]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [72]:
data_dir = f"{path}/data"

full_train_dataset = datasets.ImageFolder(root=data_dir, transform=train_transforms)
full_val_dataset = datasets.ImageFolder(root=data_dir, transform=val_transforms)

targets = full_train_dataset.targets
indices = list(range(len(targets)))

train_indices, test_indices = train_test_split(indices, test_size=0.2, stratify=targets, random_state=42)

train_dataset = Subset(dataset=full_train_dataset, indices=train_indices)
val_dataset = Subset(dataset=full_val_dataset, indices=test_indices)

train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False, num_workers=2)
full_train_dataset.class_to_idx


{'Cassava___bacterial_blight': 0,
 'Cassava___brown_streak_disease': 1,
 'Cassava___green_mottle': 2,
 'Cassava___healthy': 3,
 'Cassava___mosaic_disease': 4}

In [73]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [74]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(in_features=num_features, out_features=5)

model.to(device)

class_weights = torch.tensor([2.0, 1.0, 1.0, 1.0, 0.17]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(params=model.fc.parameters(), lr=1e-3)
accuracy = Accuracy(task="multiclass", num_classes = 5)
accuracy = accuracy.to(device)


In [75]:
def train_model(model, criterion, optimizer, num_epochs=5, initial_best_acc=0.0):
    dataloaders = {
        "train": train_loader,
        "val": val_loader
    }

    dataset_length = {
        "train": len(train_loader.dataset),
        "val": len(val_loader.dataset)
    }

    best_acc = initial_best_acc
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        print(f"Epoch: {epoch+1}/{num_epochs}")
        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            accuracy.reset()

            for X_batch, y_batch in dataloaders[phase]:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                with torch.set_grad_enabled(phase=="train"):
                    y_preds = model(X_batch)
                    loss = criterion(y_preds, y_batch)

                    if phase == "train":
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * X_batch.size(0)
                accuracy.update(y_preds, y_batch)

            epoch_loss = running_loss / dataset_length[phase]
            epoch_acc = accuracy.compute().item()

            print(f"{phase}:: loss: {epoch_loss:.4f}, accuracy: {epoch_acc:.2f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_weights = copy.deepcopy(model.state_dict())


    print('\n')
    model.load_state_dict(best_weights)
    return model, best_acc



In [76]:
model, best_acc = train_model(model=model, criterion=criterion, optimizer=optimizer, num_epochs=5)

Epoch: 1/5
train:: loss: 1.2495, accuracy: 0.57
val:: loss: 1.1309, accuracy: 0.62
Epoch: 2/5
train:: loss: 1.1519, accuracy: 0.61
val:: loss: 1.1197, accuracy: 0.67
Epoch: 3/5
train:: loss: 1.1182, accuracy: 0.62
val:: loss: 1.1113, accuracy: 0.54
Epoch: 4/5
train:: loss: 1.1073, accuracy: 0.62
val:: loss: 1.0913, accuracy: 0.67
Epoch: 5/5
train:: loss: 1.1075, accuracy: 0.62
val:: loss: 1.1344, accuracy: 0.61




In [ ]:
for name, child in model.named_children():
    if name in ["layer3", "layer4"]:
        for param in child.parameters():
            param.requires_grad = True

optimizer_fine = torch.optim.Adam([
    {"params": model.layer3.parameters(), 'lr': 1e-5},
    {"params": model.layer4.parameters(), 'lr': 1e-5},
    {"params": model.fc.parameters(), 'lr': 1e-4}
])

model, best_final_acc = train_model(model=model, criterion=criterion, optimizer=optimizer_fine, num_epochs=5, initial_best_acc=best_acc)

Epoch: 1/5
train:: loss: 1.0175, accuracy: 0.66
val:: loss: 0.9695, accuracy: 0.67
Epoch: 2/5
train:: loss: 0.9302, accuracy: 0.69
val:: loss: 0.9224, accuracy: 0.70
Epoch: 3/5
train:: loss: 0.9003, accuracy: 0.70
val:: loss: 0.8957, accuracy: 0.73
Epoch: 4/5
train:: loss: 0.8667, accuracy: 0.72
val:: loss: 0.8950, accuracy: 0.76
Epoch: 5/5
